# 2. Station features for clustering

Collapses roughly 267,000 daily records into one row per station (N = 434), describing volume, variability and the weekly usage pattern.

## 2.1 Load the processed data

In [1]:
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, "../src")
import tfl

raw = pd.read_csv("../data/processed/TfL_Data_Processed.csv", parse_dates=["TravelDate"])

# The September 2025 strike is a one-off network event, not station behaviour.
strike = raw["TravelDate"].between(*tfl.STRIKE)
print("strike-week rows excluded:", strike.sum())
raw = raw[~strike]
raw.head()

strike-week rows excluded: 3014


,TravelDate,DayOfWeek,Station,EntryTapCount,ExitTapCount,DayNum,Month,Year,IsWeekend
0,2024-01-01,Monday,Abbey Road DLR,395,375,0,1,2024,False
1,2024-01-01,Monday,Abbey Wood,5898,5963,0,1,2024,False
2,2024-01-01,Monday,Acton Central,609,474,0,1,2024,False
3,2024-01-01,Monday,Acton Main Line,1717,1710,0,1,2024,False
4,2024-01-01,Monday,Acton Town,2928,3334,0,1,2024,False


## 2.2 Station volume

In [2]:
mean_traffic = (raw[["Station", "EntryTapCount"]]
      .groupby("Station")
      .agg("mean")
      .assign(LogMeanEntryCount=lambda d: np.log10(d["EntryTapCount"]))
      )[["LogMeanEntryCount"]]

print(mean_traffic.head(), "\n")
mean_traffic.describe()

                 LogMeanEntryCount
Station                           
Abbey Road DLR            2.921015
Abbey Wood                4.189223
Acton Central             3.318122
Acton Main Line           3.585486
Acton Town                3.833960 



,LogMeanEntryCount
count,434.000000
mean,3.694018
std,0.456117
min,2.295810
25%,3.387809
50%,3.677139
75%,3.944739
max,5.024042


## 2.3 Variability

In [3]:
traffic_variability = (raw[["Station", "EntryTapCount"]]
      .groupby("Station")
      .agg("std")
      .assign(CoefVariation=lambda d: d["EntryTapCount"] / raw.groupby("Station")["EntryTapCount"].mean())
      .drop("EntryTapCount", axis="columns")
      )

print(traffic_variability.head(), "\n")
traffic_variability.describe()

                 CoefVariation
Station                       
Abbey Road DLR        0.300102
Abbey Wood            0.236017
Acton Central         0.294328
Acton Main Line       0.231024
Acton Town            0.242458 



,CoefVariation
count,434.000000
mean,0.263601
std,0.075501
min,0.132687
25%,0.211920
50%,0.247361
75%,0.299569
max,0.728754


## 2.4 Weekly usage pattern

In [4]:
day_rename_map = {0: "Monday",
                  1: "Tuesday",
                  2: "Wednesday",
                  3: "Thursday",
                  4: "Friday",
                  5: "Saturday",
                  6: "Sunday"}

weekly_share = (raw[["Station", "DayNum", "EntryTapCount"]]
                .groupby(["Station", "DayNum"])
                .mean()
                .assign(WeeklyShare=lambda d: d["EntryTapCount"] / d.groupby("Station")["EntryTapCount"].sum())
                .pivot_table(index="Station", columns="DayNum", values="WeeklyShare")
                .rename(day_rename_map, axis="columns")
                )
weekly_share

DayNum,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
Station,,,,,,,
Abbey Road DLR,0.146913,0.165332,0.165950,0.170579,0.165609,0.104947,0.080669
Abbey Wood,0.142919,0.163532,0.164997,0.168141,0.153408,0.123698,0.083304
Acton Central,0.141377,0.161683,0.161660,0.165370,0.164737,0.132763,0.072410
Acton Main Line,0.139318,0.156698,0.161074,0.168839,0.156015,0.137632,0.080423
Acton Town,0.138364,0.157642,0.160028,0.161692,0.158009,0.128119,0.096146
...,...,...,...,...,...,...,...
Woodgrange Park,0.144641,0.156482,0.158562,0.158604,0.160717,0.135939,0.085054
Woodside Park,0.140486,0.165125,0.166910,0.170083,0.151516,0.122469,0.083411
Woolwich Arsenal DLR,0.147351,0.160050,0.156484,0.154325,0.157135,0.122035,0.102620


In [5]:
d = (weekly_share
     .reset_index()
     .melt(id_vars="Station", var_name="Weekday", value_name="value")
     )

is_weekend = d["Weekday"].isin(["Saturday", "Sunday"])
weekday_avg = d[~is_weekend].groupby("Station")["value"].mean()
weekend_avg = d[is_weekend].groupby("Station")["value"].mean()

weekly_balance = (weekday_avg / weekend_avg).rename("WeekdayWeekendBalance")
weekly_balance

Station
Abbey Road DLR          1.754985
Abbey Wood              1.532354
Acton Central           1.549571
Acton Main Line         1.434396
Acton Town              1.383609
                          ...   
Woodgrange Park         1.410012
Woodside Park           1.542885
Woolwich Arsenal DLR    1.380508
Woolwich Arsenal NR     1.313847
Woolwich EL             1.294761
Name: WeekdayWeekendBalance, Length: 434, dtype: float64

## 2.5 Join the features

In [6]:
station_features = (mean_traffic
                    .join(traffic_variability, how="inner")
                    .join(weekly_share, how="inner")
                    .join(weekly_balance, how="inner"))
station_features.head()

,LogMeanEntryCount,CoefVariation,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,WeekdayWeekendBalance
Station,,,,,,,,,,
Abbey Road DLR,2.921015,0.300102,0.146913,0.165332,0.165950,0.170579,0.165609,0.104947,0.080669,1.754985
Abbey Wood,4.189223,0.236017,0.142919,0.163532,0.164997,0.168141,0.153408,0.123698,0.083304,1.532354
Acton Central,3.318122,0.294328,0.141377,0.161683,0.161660,0.165370,0.164737,0.132763,0.072410,1.549571
Acton Main Line,3.585486,0.231024,0.139318,0.156698,0.161074,0.168839,0.156015,0.137632,0.080423,1.434396
Acton Town,3.833960,0.242458,0.138364,0.157642,0.160028,0.161692,0.158009,0.128119,0.096146,1.383609


In [7]:
station_features.describe()

,LogMeanEntryCount,CoefVariation,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,WeekdayWeekendBalance
count,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000
mean,3.694018,0.263601,0.138656,0.160113,0.162979,0.166326,0.153773,0.128328,0.089824,1.491958
std,0.456117,0.075501,0.008323,0.011834,0.010971,0.010652,0.007244,0.020928,0.018018,0.375327
min,2.295810,0.132687,0.100603,0.090804,0.107113,0.131782,0.117663,0.050822,0.033306,0.759832
25%,3.387809,0.211920,0.134252,0.154256,0.157382,0.160945,0.149875,0.116906,0.078346,1.289828
50%,3.677139,0.247361,0.139334,0.160126,0.163064,0.165662,0.155000,0.128516,0.088545,1.443685
75%,3.944739,0.299569,0.143959,0.165196,0.168354,0.171269,0.158308,0.139039,0.099525,1.612870
max,5.024042,0.728754,0.176682,0.207786,0.210923,0.219602,0.168921,0.212974,0.164981,3.992175


In [8]:
missing_stations = len(station_features) - raw["Station"].nunique()

if missing_stations == 0:
    print("No stations dropped.")
else:
    print(missing_stations, "stations dropped.")

No stations dropped.


In [9]:
station_features.corr()
# Tuesday, Wednesday, Thursday, Saturday and Sunday each correlate with
# WeekdayWeekendBalance at |r| > 0.8, so including every day would let the weekly
# pattern dominate the distance metric. Friday is nearly independent of the balance
# and is kept as a separate feature.

,LogMeanEntryCount,CoefVariation,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,WeekdayWeekendBalance
LogMeanEntryCount,1.000000,-0.322321,-0.307490,-0.229631,-0.222822,-0.154511,-0.255162,0.278810,0.298616,-0.221172
CoefVariation,-0.322321,1.000000,0.183608,0.476975,0.547945,0.554420,-0.126753,-0.468779,-0.464013,0.594285
Monday,-0.307490,0.183608,1.000000,0.570963,0.464658,0.406295,0.219445,-0.789443,-0.531319,0.633647
Tuesday,-0.229631,0.476975,0.570963,1.000000,0.956295,0.874789,-0.276041,-0.878957,-0.888034,0.897731
Wednesday,-0.222822,0.547945,0.464658,0.956295,1.000000,0.904577,-0.270514,-0.844221,-0.897012,0.883503
Thursday,-0.154511,0.554420,0.406295,0.874789,0.904577,1.000000,-0.315899,-0.807402,-0.839351,0.869555
Friday,-0.255162,-0.126753,0.219445,-0.276041,-0.270514,-0.315899,1.000000,0.039817,-0.016909,-0.116298
Saturday,0.278810,-0.468779,-0.789443,-0.878957,-0.844221,-0.807402,0.039817,1.000000,0.755768,-0.895037
Sunday,0.298616,-0.464013,-0.531319,-0.888034,-0.897012,-0.839351,-0.016909,0.755768,1.000000,-0.847955
WeekdayWeekendBalance,-0.221172,0.594285,0.633647,0.897731,0.883503,0.869555,-0.116298,-0.895037,-0.847955,1.000000


In [10]:
station_features_selected = station_features[["LogMeanEntryCount", "CoefVariation", "Friday", "WeekdayWeekendBalance"]]
station_features_selected.head()

,LogMeanEntryCount,CoefVariation,Friday,WeekdayWeekendBalance
Station,,,,
Abbey Road DLR,2.921015,0.300102,0.165609,1.754985
Abbey Wood,4.189223,0.236017,0.153408,1.532354
Acton Central,3.318122,0.294328,0.164737,1.549571
Acton Main Line,3.585486,0.231024,0.156015,1.434396
Acton Town,3.833960,0.242458,0.158009,1.383609


## 2.6 Save

The features are saved in real units and standardised in notebook 3.

In [11]:
path = "../data/processed/TfL_Station_Features.csv"
station_features_selected.to_csv(path, index=True)
print("Output saved to:", path)

Output saved to: ../data/processed/TfL_Station_Features.csv
